# 33 — DSPy RAG

Build a RAG pipeline using **DSPy Signatures** instead of hand-crafted prompts. Compile the module with BootstrapFewShot to auto-discover few-shot examples, then compare base vs compiled answers.

**What you'll learn**
- Why DSPy replaces prompt engineering with compiled programs
- `dspy.Signature`: typed input/output fields that define the task
- `dspy.ChainOfThought`: adds reasoning trace before the answer
- `dspy.Module`: composable Python class that holds DSPy predictors
- `BootstrapFewShot`: automatically selects few-shot examples from a labeled trainset
- Base vs compiled: what changes after compilation

**Contrast:** LangChain LCEL — hand-write `PromptTemplate | ChatOpenAI | StrOutputParser`; DSPy — declare `Signature` fields and let the compiler tune the prompts.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q dspy-ai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
if not IN_COLAB:
    load_dotenv()
if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. DSPy vs LangChain LCEL — the key idea ─────────────────────────────────
#
# LangChain LCEL:  you write the prompt manually
#   template = PromptTemplate.from_template("Answer: {question}\nContext: {context}")
#   chain = template | ChatOpenAI() | StrOutputParser()
#
# DSPy:  you declare WHAT the task is; the compiler writes/tunes the prompt for you
#   class GenerateAnswer(dspy.Signature):
#       context: str = dspy.InputField()
#       question: str = dspy.InputField()
#       answer: str = dspy.OutputField()
#
# BootstrapFewShot runs the module on the trainset and picks examples where
# the metric passes — those become automatic few-shot demonstrations.

print("LangChain LCEL: PromptTemplate | model | parser  (you write the prompt)")
print("DSPy:           Signature + Module + compiler   (compiler writes the prompt)")

In [ ]:
# ── 4. Build the DSPy RAG pipeline ───────────────────────────────────────────
import dspy
from dspy.teleprompt import BootstrapFewShot

dspy.configure(lm=dspy.LM("openai/gpt-4o-mini", temperature=0))

DOCS = [
    "DSPy was created by Stanford NLP and released in 2023.",
    "DSPy replaces hand-crafted prompts with compiled, auto-optimized signatures.",
    "DSPy Signatures define typed input and output fields using Python class syntax.",
    "DSPy ChainOfThought adds step-by-step reasoning before producing the final answer.",
    "DSPy BootstrapFewShot automatically selects few-shot examples from a labeled trainset.",
    "MIPROv2 is a DSPy optimizer that jointly tunes instructions and few-shot examples.",
    "DSPy modules are composable Python classes that inherit from dspy.Module.",
    "LangGraph was created by LangChain Inc. and released in January 2024.",
    "LangChain LCEL uses the | pipe operator to chain prompts, models, and parsers.",
    "LangGraph StateGraph compiles into a runnable that can be invoked or streamed.",
]

TRAINSET = [
    dspy.Example(question="Who created DSPy?", answer="Stanford NLP").with_inputs("question"),
    dspy.Example(question="What does DSPy ChainOfThought do?",
                 answer="adds step-by-step reasoning before the final answer").with_inputs("question"),
    dspy.Example(question="When was LangGraph released?",
                 answer="January 2024").with_inputs("question"),
    dspy.Example(question="What is BootstrapFewShot?",
                 answer="selects few-shot examples automatically from a trainset").with_inputs("question"),
]


class GenerateAnswer(dspy.Signature):
    """Answer a question given relevant context from a knowledge base."""
    context: str = dspy.InputField(desc="relevant facts from the knowledge base")
    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="1-2 sentences, factual and concise")


def keyword_retrieve(question: str, k: int = 3) -> str:
    words = set(question.lower().split())
    scored = [(sum(w in d.lower() for w in words), d) for d in DOCS]
    scored.sort(reverse=True)
    top = [d for _, d in scored[:k] if _ > 0] or DOCS[:k]
    return "\n".join(top)


class RAG(dspy.Module):
    def __init__(self):
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question: str):
        context = keyword_retrieve(question)
        return self.generate(context=context, question=question)


def validate_answer(example, pred, trace=None) -> bool:
    return bool(pred.answer and len(pred.answer.strip()) > 10)


print("Compiling RAG with BootstrapFewShot...")
base_rag = RAG()
compiled_rag = BootstrapFewShot(
    metric=validate_answer, max_bootstrapped_demos=2, max_labeled_demos=2
).compile(RAG(), trainset=TRAINSET)
print("Done. Base and compiled RAG ready.")

In [ ]:
# ── 5. Run base vs compiled ───────────────────────────────────────────────────
QUESTIONS = [
    "Who created DSPy and when?",
    "How does DSPy differ from LangChain LCEL?",
    "What is the role of MIPROv2 in DSPy?",
]

for q in QUESTIONS:
    b = base_rag(question=q)
    c = compiled_rag(question=q)
    print(f"Q: {q}")
    print(f"  [base]     {b.answer}")
    print(f"  [compiled] {c.answer}")
    print()

## Exercises

**Exercise 1 — Change the Signature:** Remove `desc=` from `GenerateAnswer.answer`. Does the answer quality change? What prompt does DSPy generate?

**Exercise 2 — Inspect the compiled prompts:** After compilation, print `compiled_rag.generate.extended_signature` to see what DSPy actually sends to the model.

**Exercise 3 — Swap to MIPROv2:** Replace `BootstrapFewShot` with `dspy.teleprompt.MIPROv2(metric=validate_answer, auto="light")` and compare the optimized instructions.

**Exercise 4 — Real embeddings retriever:** Replace `keyword_retrieve` with a Chroma-backed retriever (see example 32) and repeat compilation. Does a better retriever change what few-shots the compiler picks?

## What's next

- **32-speculative-rag** — draft first, verify claims, revise only what's wrong
- **25-adaptive-rag** — classify query type and pick the cheapest retrieval strategy
- **17-corrective-rag** — retrieve first, then score and correct